# Exercice 6 : Synthèse : rapport de surface d'attaque

> **En une phrase :** on assemble les exercices 3 et 4 en **un seul tableau** priorisé, puis on en tire **trois recommandations** pour le défenseur.

**Le scénario.** Vous êtes l'analyste chargé de la **gestion de la surface d'attaque** de l'UQAC. Votre gestionnaire veut une page, pas 160 lignes : quoi corriger en premier ?

**Le plan.**
1. Collecter (passif) : noms, IP, ports, logiciels.
2. Marquer les signaux de risque.
3. Prioriser.
4. Rédiger le rapport.

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import requests

def dns(nom, type_enr="A"):
    """Pose une question DNS publique (via dns.google) et renvoie la liste des réponses."""
    codes = {"A": 1, "NS": 2, "CNAME": 5, "MX": 15, "TXT": 16}
    rep = requests.get("https://dns.google/resolve", params={"name": nom, "type": type_enr}, timeout=10).json()
    return [r["data"].strip('"') for r in rep.get("Answer", []) if r["type"] == codes[type_enr]]

import time

def sous_domaines(domaine):
    """Liste les noms trouvés dans les journaux Certificate Transparency (crt.sh)."""
    for essai in range(3):                       # crt.sh est parfois lent : on réessaie
        try:
            certs = requests.get("https://crt.sh/", params={"q": "%." + domaine, "output": "json"}, timeout=90).json()
            break
        except Exception:
            print("crt.sh ne répond pas, nouvel essai…"); time.sleep(5)
    else:
        raise RuntimeError("crt.sh indisponible : réessayez dans une minute.")
    noms = set()
    for c in certs:
        for n in c["name_value"].split("\n"):
            n = n.strip().lower().removeprefix("*.")
            if n.endswith(domaine):
                noms.add(n)
    return certs, sorted(noms)

from concurrent.futures import ThreadPoolExecutor
import pandas as pd
print('Prêt')

Prêt


## Étape 1 : Collecter (passif)
**Ce qu'on fait :** exactement la collecte des ex. 3 et 4, en une seule cellule. *(Environ 1 minute.)*

In [2]:
domaine = "uqac.ca"      # <-- MODIFIABLE
bloc_interne = "132.212."  # le bloc d'adresses qui appartient à l'organisation

_, noms = sous_domaines(domaine)
with ThreadPoolExecutor(16) as pool:
    resolutions = dict(zip(noms, pool.map(dns, noms)))

def shodan(ip):
    r = requests.get(f"https://internetdb.shodan.io/{ip}", timeout=10)
    return r.json() if r.status_code == 200 else {}

ips = sorted({ip for l in resolutions.values() for ip in l})
with ThreadPoolExecutor(8) as pool:
    infos = dict(zip(ips, pool.map(shodan, ips)))

lignes = []
for nom, liste in resolutions.items():
    for ip in liste:
        d = infos.get(ip, {})
        lignes.append({"nom": nom, "ip": ip, "ports": d.get("ports", []), "cpes": d.get("cpes", [])})
rapport = pd.DataFrame(lignes)
print(f"{len(noms)} noms trouvés, {rapport['nom'].nunique()} publics, {len(ips)} IP")

crt.sh ne répond pas, nouvel essai…


164 noms trouvés, 43 publics, 42 IP


**👀 Ce qu'on doit voir :** ≈ **163 noms**, ≈ **42 publics**, ≈ **42 à 46 IP**.

## Étape 2 : Marquer les signaux de risque
**Ce qu'on fait :** chaque règle traduit une leçon du cours en une colonne vrai/faux.

In [3]:
PORTS_WEB = {80, 443}

rapport["nom_dev_test"]   = rapport["nom"].str.contains(r"dev|test|staging|preprod")          # ex. 3
rapport["chez_un_tiers"]  = ~rapport["ip"].str.startswith(bloc_interne)                       # ex. 4
rapport["ports_non_web"]  = rapport["ports"].apply(lambda p: sorted(set(p) - PORTS_WEB))       # ex. 4-5
rapport["version_exposee"] = rapport["cpes"].apply(lambda c: any(x.count(":") >= 4 for x in c))  # ex. 5

rapport[["nom_dev_test", "chez_un_tiers", "version_exposee"]].sum()

nom_dev_test        2
chez_un_tiers      34
version_exposee    18
dtype: int64

**👀 Ce qu'on doit voir :** **2** serveurs « -dev » encore joignables (sur les 15 de l'ex. 3), ≈ **33 lignes** chez un tiers, et ≈ **17** qui annoncent leur version exacte.

## Étape 3 : Prioriser
**Ce qu'on fait :** un score simple. Ce n'est pas une science exacte : c'est un **ordre de travail** pour l'équipe.

In [4]:
# <-- MODIFIABLE : les poids reflètent votre jugement d'analyste
rapport["score"] = (3 * rapport["nom_dev_test"]
                    + 2 * rapport["version_exposee"]
                    + 1 * rapport["chez_un_tiers"]
                    + rapport["ports_non_web"].apply(len).clip(upper=3))

colonnes = ["nom", "ip", "score", "nom_dev_test", "chez_un_tiers", "version_exposee", "ports_non_web"]
rapport.sort_values("score", ascending=False)[colonnes].head(10)

,nom,ip,score,nom_dev_test,chez_un_tiers,version_exposee,ports_non_web
12,constellation-dev.uqac.ca,132.209.15.176,6,True,True,True,[]
16,desiir.uqac.ca,68.71.54.196,6,False,True,True,"[21, 53, 110, 143, 465, 587, 993, 995, 2079, 2..."
9,cisd.uqac.ca,68.71.54.196,6,False,True,True,"[21, 53, 110, 143, 465, 587, 993, 995, 2079, 2..."
45,sdeir-dev.uqac.ca,132.209.15.176,6,True,True,True,[]
23,img.envois.uqac.ca,104.17.157.243,4,False,True,False,"[2052, 2053, 2082, 2083, 2086, 2087, 2095, 209..."
19,img.envois.uqac.ca,104.17.159.243,4,False,True,False,"[2052, 2053, 2082, 2083, 2086, 2087, 2095, 209..."
21,img.envois.uqac.ca,104.17.158.243,4,False,True,False,"[2052, 2053, 2082, 2083, 2086, 2087, 2095, 209..."
22,img.envois.uqac.ca,104.17.155.243,4,False,True,False,"[2052, 2053, 2082, 2083, 2086, 2087, 2095, 209..."
20,img.envois.uqac.ca,104.17.156.243,4,False,True,False,"[2052, 2053, 2082, 2083, 2086, 2087, 2095, 209..."
8,bibliotheque.uqac.ca,35.183.221.217,3,False,True,True,[]


**👀 Ce qu'on doit voir :** en tête (score 6) : deux sites sur un **hébergement mutualisé** (beaucoup de ports ouverts) et deux serveurs **« -dev »** qui annoncent leur version. Juste après (score 4) : des serveurs Cloudflare.

**❓ Question pour la classe :** les serveurs Cloudflare (`img.envois…`) ont 11 ports ouverts et un score de 4. Est-ce vraiment un risque pour l'UQAC ?

*Réponse : probablement pas : ces ports sont ouverts sur **tous** les serveurs Cloudflare, c'est leur configuration standard. Un score automatique **doit être relu** par un humain, comme les CVE de l'ex. 4.*

## Étape 4 : Rédiger le rapport
**Ce qu'on fait :** on ajoute la sécurité du courriel et le WHOIS, puis on écrit le rapport d'une page.

In [5]:
dmarc = dns("_dmarc." + domaine, "TXT")
rdap = requests.get(f"https://rdap.ca.fury.ca/rdap/domain/{domaine}", timeout=15).json()
contacts_nommes = sum(1 for e in rdap.get("entities", []) if set(e["roles"]) & {"administrative", "technical"}
                      for c in e.get("vcardArray", [None, []])[1] if c[0] == "fn" and c[3])

print(f"RAPPORT DE SURFACE D'ATTAQUE : {domaine}")
print("=" * 50)
print(f"Noms dans les certificats publics : {len(noms)} (dont {rapport['nom'].nunique()} joignables)")
print(f"Noms hébergés chez un tiers       : {rapport.loc[rapport['chez_un_tiers'], 'nom'].nunique()}")
print(f"Noms dev/test encore publics      : {rapport.loc[rapport['nom_dev_test'], 'nom'].nunique()}")
print(f"Serveurs qui annoncent leur version : {rapport.loc[rapport['version_exposee'], 'nom'].nunique()}")
print(f"DMARC                              : {dmarc[0] if dmarc else 'ABSENT'}")
print(f"Contacts nommés dans le WHOIS      : {contacts_nommes}")
print()
print("TROIS RECOMMANDATIONS")
print("1. Inventorier les sous-domaines chez des tiers ; supprimer les entrées DNS orphelines (prise de contrôle).")
print("2. Retirer de l'Internet public les serveurs dev/test ; utiliser une autorité de certification interne.")
print("3. Remplacer les noms du WHOIS par une adresse de rôle ; masquer les versions dans les bannières.")

RAPPORT DE SURFACE D'ATTAQUE : uqac.ca
Noms dans les certificats publics : 164 (dont 43 joignables)
Noms hébergés chez un tiers       : 21
Noms dev/test encore publics      : 2
Serveurs qui annoncent leur version : 16
DMARC                              : v=DMARC1; p=reject;sp=quarantine;rua=mailto:dmarc_agg@vali.email;fo=1;
Contacts nommés dans le WHOIS      : 2

TROIS RECOMMANDATIONS
1. Inventorier les sous-domaines chez des tiers ; supprimer les entrées DNS orphelines (prise de contrôle).
2. Retirer de l'Internet public les serveurs dev/test ; utiliser une autorité de certification interne.
3. Remplacer les noms du WHOIS par une adresse de rôle ; masquer les versions dans les bannières.


**👀 Ce qu'on doit voir :** un rapport d'une dizaine de lignes : ≈ 163 noms, ≈ 20 chez des tiers, ≈ 15 qui annoncent leur version, DMARC **p=reject** (bon point), **2** contacts nommés, et trois recommandations concrètes.

## Ce qu'il faut retenir
- La reconnaissance sert **autant au défenseur qu'à l'attaquant** : c'est la **gestion de la surface d'attaque**.
- Un rapport utile est **court**, **priorisé** et finit par des **actions**.
- Tout ici était **passif** et **gratuit**. Si un analyste le fait en une heure, un attaquant aussi : autant le faire **avant lui**.

**Séance suivante (9) :** on change de côté. Le scan nmap de l'ex. 5, à quoi ressemble-t-il dans les journaux du défenseur ?